In [1]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity, rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Gene Exp

In [2]:
cell_info = pd.read_excel(
    "/Users/inouey2/Downloads/Cell_Lines_Details.xlsx",
    usecols=["Sample Name", "COSMIC identifier"],
)
cell_info.head()

/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Sample Name,COSMIC identifier
0,A253,906794.0
1,BB30-HNC,753531.0
2,BB49-HNC,753532.0
3,BHY,753535.0
4,BICR10,1290724.0


In [3]:
gene_exp = pd.read_table("/Users/inouey2/Downloads/Cell_line_RMA_proc_basalExp.txt.zip")
gene_exp.index = list(gene_exp["GENE_SYMBOLS"])
gene_exp = gene_exp[~gene_exp.index.isna()]
gene_exp = gene_exp.drop(["GENE_SYMBOLS", "GENE_title"], axis=1)
gene_exp.columns = [float(i[5:]) for i in gene_exp.columns]
gene_exp = gene_exp[sorted(set(gene_exp.columns) & set(cell_info["COSMIC identifier"]))]
gene_exp.columns = [
    cell_info[cell_info["COSMIC identifier"] == i]["Sample Name"].iloc[0]
    for i in gene_exp.columns
]
gene_exp = gene_exp.T
gene_exp = gene_exp.rename(index={"GAK": "GAK_cell"})
gene_exp.head()

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,LINC00514,OR1D5,ZNF234,MYH4,LINC00526,PPY2,KRT18P55,POLRMTP1,UBL5P2,TBC1D3P5
MC-CAR,3.238273,2.982254,10.235491,4.856061,4.078870,9.116236,3.658590,6.145475,5.042464,5.438402,...,3.103752,3.724013,3.981948,2.823245,5.866047,3.095716,3.274367,3.056214,9.446305,3.530871
PFSK-1,7.780713,2.753253,9.960137,4.351073,3.716740,3.222277,8.221606,3.823474,4.756228,5.805642,...,3.175476,3.779354,4.504481,2.690651,3.347520,3.230713,3.102091,3.169188,9.810430,3.266915
A673,7.301344,2.890533,9.922489,4.125088,3.678987,3.096576,3.588391,4.809305,4.951782,5.089165,...,3.299300,3.762301,4.177345,2.499803,5.054260,3.003521,3.068187,3.135479,9.073222,3.098364
ES3,8.690198,3.091473,9.992487,4.572198,3.333385,3.320793,3.159487,3.515105,5.446361,5.348338,...,3.251885,2.929491,4.702208,2.489674,5.097089,3.114744,3.119647,3.194925,9.013220,3.074187
ES5,8.233101,2.824687,10.015884,4.749715,3.839433,3.142754,5.329830,3.272124,5.538055,6.428482,...,3.081750,3.226083,4.666295,2.491254,6.261573,3.031862,3.322455,2.813440,8.893197,3.266184


# Drug Response

In [4]:
drug_response = pd.read_excel(
    "/Users/inouey2/Downloads/GDSC2_fitted_dose_response_27Oct23.xlsx",
    usecols=["DRUG_NAME", "CELL_LINE_NAME", "Z_SCORE"],
)
drug_response["CELL_LINE_NAME"] = drug_response["CELL_LINE_NAME"].replace(
    "GAK", "GAK_cell"
)
drug_response

,CELL_LINE_NAME,DRUG_NAME,Z_SCORE
0,PFSK-1,Camptothecin,0.433123
1,A673,Camptothecin,-1.421100
2,ES5,Camptothecin,-0.599569
3,ES7,Camptothecin,-1.516647
4,EW-11,Camptothecin,-0.807232
...,...,...,...
242031,SNU-175,N-acetyl cysteine,0.156872
242032,SNU-407,N-acetyl cysteine,-1.626959
242033,SNU-61,N-acetyl cysteine,0.608442
242034,SNU-C5,N-acetyl cysteine,0.809684


In [5]:
l = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
    ["NAME", "SMILES"]
]
l

,NAME,SMILES
0,p-Toluquinone,CC1=CC(=O)C=CC1=O
1,4-Amino-3-pentadecylphenol,CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N
2,3-(Dimethylamino)propiophenone hydrochloride,CN(C)CCC(=O)C1=CC=CC=C1.Cl
3,Cycloheximide,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
4,Cycloheximide,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
...,...,...
23186,"N-(5-piperazin-1-ylpyridin-2-yl)-4-(7-thia-9,1...",C1CC2=C(C1)SC3=NC=NC(=C23)NC4=C(NN=C4)C(=O)NC5...
23187,"(11Z)-3,5-diethyl-11-(2-oxo-1-prop-2-enylindol...",CCN1C2C(N(C1=O)CC)N3C(=O)C(=C4C5=CC=CC=C5N(C4=...
23188,2-[(E)-benzylideneamino]-4-phenyl-4H-benzo[h]c...,C1=CC=C(C=C1)C=NC2=C(C(C3=C(O2)C4=CC=CC=C4C=C3...
23189,Naporafenib,CC1=C(C=C(C=C1)NC(=O)C2=CC(=NC=C2)C(F)(F)F)C3=...


In [6]:
df = drug_response[drug_response["DRUG_NAME"].isin(l["NAME"])]
df = df[
    df["CELL_LINE_NAME"].isin(set(drug_response.CELL_LINE_NAME) & set(gene_exp.index))
]
df

,CELL_LINE_NAME,DRUG_NAME,Z_SCORE
0,PFSK-1,Camptothecin,0.433123
1,A673,Camptothecin,-1.421100
2,ES5,Camptothecin,-0.599569
3,ES7,Camptothecin,-1.516647
4,EW-11,Camptothecin,-0.807232
...,...,...,...
219498,RCC-JW,Uprosertib,-0.206009
219499,MM1S,Uprosertib,-2.693080
219500,SNU-175,Uprosertib,-1.093492
219501,SNU-407,Uprosertib,-1.036300


# Zero shot prediction

In [7]:
unique_drugs = df["DRUG_NAME"].unique()
unique_cells = df["CELL_LINE_NAME"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[
    (df["DRUG_NAME"].isin(train_drugs)) & (df["CELL_LINE_NAME"].isin(train_cells))
]
val_df = df[(df["DRUG_NAME"].isin(val_drugs)) & (df["CELL_LINE_NAME"].isin(val_cells))]
test_df = df[
    (df["DRUG_NAME"].isin(test_drugs)) & (df["CELL_LINE_NAME"].isin(test_cells))
]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Z_SCORE"] > 0]
    negative = df[df["Z_SCORE"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["DRUG_NAME", "CELL_LINE_NAME"]]
y_train = np.array(train_df["Z_SCORE"] > 0, dtype=float)

X_val = val_df[["DRUG_NAME", "CELL_LINE_NAME"]]
y_val = np.array(val_df["Z_SCORE"] > 0, dtype=float)

X_test = test_df[["DRUG_NAME", "CELL_LINE_NAME"]]
y_test = np.array(test_df["Z_SCORE"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

X_train.to_csv("../GDSC2_data/train.csv", index=False)
X_test.to_csv("../GDSC2_data/test.csv", index=False)
X_val.to_csv("../GDSC2_data/val.csv", index=False)

np.save("../GDSC2_data/train_labels.npy", y_train)
np.save("../GDSC2_data/test_labels.npy", y_test)
np.save("../GDSC2_data/val_labels.npy", y_val)

Train:
(13868, 2) (13868,)
Percentage: 62.50%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(4138, 2) (4138,)
Percentage: 18.65%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(4184, 2) (4184,)
Percentage: 18.86%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 22190
Ratio (Train:Validation:Test): 13868:4138:4184
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [8]:
drug_response = df.pivot_table(
    columns="CELL_LINE_NAME", index="DRUG_NAME", values="Z_SCORE"
)
drug_response.index = list(drug_response.index)
drug_response.columns = list(drug_response.columns)
drug_response = drug_response.fillna(0)
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.499758,-0.287872,-0.044328,0.445754,-0.018312,-0.056024,-1.007905,-0.133606,0.039340,0.332709,...,-0.630538,-1.313438,0.462079,0.219408,0.967360,-0.212762,-2.278620,0.570567,0.332668,2.724350
Afuresertib,-0.576528,-1.027199,-1.251724,0.776892,0.483202,1.035221,0.702506,-1.467521,-0.995829,0.279814,...,-3.524001,-2.226865,1.767767,0.160727,0.003439,-0.288511,0.620736,1.031922,-0.303296,0.173684
Alisertib,0.744968,1.227995,0.106415,1.396777,-0.067135,0.703375,0.230105,-0.070375,0.175436,1.116507,...,0.747585,-1.310767,1.455871,-0.736767,-0.112505,0.413056,2.098072,0.280696,1.140913,0.834823
Alpelisib,-0.889693,-1.081335,-0.328515,0.994883,0.587874,1.204131,0.314342,1.214618,-0.389851,0.236610,...,-0.565499,-1.668067,0.013511,-0.751235,1.542154,-0.815384,-0.349031,0.023554,0.670297,0.647354
Avagacestat,0.284354,0.642823,0.267475,-1.214956,0.663721,0.133054,-0.413084,0.572785,0.491717,0.904438,...,-0.729761,-0.043942,1.879616,0.842820,-0.229219,-0.914654,1.669520,0.602683,0.953398,0.132863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.870938,-0.024717,0.247641,0.445747,-0.277552,-0.435486,-0.148106,-0.020043,-1.006031,1.314591,...,-1.588061,0.000000,1.000575,-1.115229,0.469415,-1.975273,1.026625,0.388924,0.568039,1.399699
Venetoclax,0.856575,0.045648,0.414053,0.104244,0.416040,-0.285095,0.744676,0.049669,-0.024445,0.239476,...,-2.341882,-1.468818,1.902045,0.263005,-0.077745,0.063586,1.714690,0.381670,0.309484,1.091016
Vinorelbine,-0.189093,-1.092180,-0.439257,0.307946,-0.064701,0.545878,0.434513,-0.945450,-0.099919,-0.240916,...,-1.516742,-1.237351,2.511912,-0.572717,0.190656,-0.305630,2.347096,0.538555,0.749441,0.788034
Vismodegib,0.451390,-0.355878,-0.326051,-0.267893,0.041656,-0.965318,-0.362449,0.687867,-0.345633,0.136962,...,-0.567849,0.000000,0.547754,-0.326945,-0.521560,-1.539789,1.777426,0.772846,-0.548637,0.506446


In [9]:
drug_response = drug_response.fillna(0)
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.499758,-0.287872,-0.044328,0.445754,-0.018312,-0.056024,-1.007905,-0.133606,0.039340,0.332709,...,-0.630538,-1.313438,0.462079,0.219408,0.967360,-0.212762,-2.278620,0.570567,0.332668,2.724350
Afuresertib,-0.576528,-1.027199,-1.251724,0.776892,0.483202,1.035221,0.702506,-1.467521,-0.995829,0.279814,...,-3.524001,-2.226865,1.767767,0.160727,0.003439,-0.288511,0.620736,1.031922,-0.303296,0.173684
Alisertib,0.744968,1.227995,0.106415,1.396777,-0.067135,0.703375,0.230105,-0.070375,0.175436,1.116507,...,0.747585,-1.310767,1.455871,-0.736767,-0.112505,0.413056,2.098072,0.280696,1.140913,0.834823
Alpelisib,-0.889693,-1.081335,-0.328515,0.994883,0.587874,1.204131,0.314342,1.214618,-0.389851,0.236610,...,-0.565499,-1.668067,0.013511,-0.751235,1.542154,-0.815384,-0.349031,0.023554,0.670297,0.647354
Avagacestat,0.284354,0.642823,0.267475,-1.214956,0.663721,0.133054,-0.413084,0.572785,0.491717,0.904438,...,-0.729761,-0.043942,1.879616,0.842820,-0.229219,-0.914654,1.669520,0.602683,0.953398,0.132863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.870938,-0.024717,0.247641,0.445747,-0.277552,-0.435486,-0.148106,-0.020043,-1.006031,1.314591,...,-1.588061,0.000000,1.000575,-1.115229,0.469415,-1.975273,1.026625,0.388924,0.568039,1.399699
Venetoclax,0.856575,0.045648,0.414053,0.104244,0.416040,-0.285095,0.744676,0.049669,-0.024445,0.239476,...,-2.341882,-1.468818,1.902045,0.263005,-0.077745,0.063586,1.714690,0.381670,0.309484,1.091016
Vinorelbine,-0.189093,-1.092180,-0.439257,0.307946,-0.064701,0.545878,0.434513,-0.945450,-0.099919,-0.240916,...,-1.516742,-1.237351,2.511912,-0.572717,0.190656,-0.305630,2.347096,0.538555,0.749441,0.788034
Vismodegib,0.451390,-0.355878,-0.326051,-0.267893,0.041656,-0.965318,-0.362449,0.687867,-0.345633,0.136962,...,-0.567849,0.000000,0.547754,-0.326945,-0.521560,-1.539789,1.777426,0.772846,-0.548637,0.506446


# Masking

In [10]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["DRUG_NAME"], row["CELL_LINE_NAME"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["DRUG_NAME"], row["CELL_LINE_NAME"]] = 0

# DTI

In [11]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="Drug Name").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Lepirudin,DB00001,NaN,46507011.0,NaN,F2,NaN
1,Cetuximab,DB00002,NaN,46507042.0,NaN,EGFR,NaN
2,Cetuximab,DB00002,NaN,46507042.0,NaN,FCGR3B,NaN
3,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QA,NaN
4,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QB,NaN


In [12]:
dti = dti[dti["Drug Name"].isin(drug_response.index)]
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1555,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,715055.0
1556,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,759856.0
1557,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,800105.0


In [13]:
print("unique drugs:", len(set(dti["Drug Name"])))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 49
unique genes: 111


In [14]:
len(set(drug_response.index) & set(dti["Drug Name"]))

49

In [15]:
len(set(gene_exp.columns) & set(dti.Gene))

106

## All drugs are in drug response.

In [16]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1555,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,715055.0
1556,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,759856.0
1557,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,800105.0
...,...,...,...,...,...,...,...
19410,Foretinib,DB12307,42642645.0,347828572.0,COC1=C(OCCCN2CCOCC2)C=C2N=CC=C(OC3=CC=C(NC(=O)...,HGF,NaN
19411,Foretinib,DB12307,42642645.0,347828572.0,COC1=C(OCCCN2CCOCC2)C=C2N=CC=C(OC3=CC=C(NC(=O)...,KDR,NaN
19413,Navitoclax,DB12340,24978538.0,347828599.0,[H][C@@](CCN1CCOCC1)(CSC1=CC=CC=C1)NC1=C(C=C(C...,BCL2,NaN
19415,Navitoclax,DB12340,24978538.0,347828599.0,[H][C@@](CCN1CCOCC1)(CSC1=CC=CC=C1)NC1=C(C=C(C...,BAD,NaN


# Selected highly variable genes

In [17]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0] == True].index)
len(variance)

1742

In [18]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  106
Top 90% variable genes:  1742
Total:  1833


# Preprocessed data dims

In [19]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(968, 1833)

In [20]:
drug_response.shape

(73, 904)

# Normalize

In [21]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,-0.183823,0.315812,1.375685,-0.831891,-0.703865,-0.519713,-1.053562,-1.144418,-1.290349,2.284174,...,-1.278763,-0.758898,0.842565,-0.882461,-0.074280,-0.214688,-0.372614,-0.763770,-1.581768,-0.893358
PFSK-1,-0.209492,0.322895,-0.328531,-0.788525,1.602383,1.927808,0.850178,-0.843973,-0.880427,-0.486080,...,0.461066,-0.816545,-0.557179,-0.623753,0.331575,-0.962710,-0.090130,-0.176769,-1.074304,0.326869
A673,-0.373586,-1.387253,-0.660569,-0.863068,-0.279611,-0.342609,1.342116,-0.454606,-0.489952,-0.444870,...,0.444578,-0.398170,0.502540,1.333584,-0.911795,1.433794,0.188239,1.216433,0.735789,-0.847409
ES3,-0.270837,-0.723571,-0.444008,-0.996405,-0.280151,-0.031850,1.091420,0.283743,-0.975739,0.279643,...,1.498397,1.513859,0.726998,2.211373,-0.509698,1.700800,0.425042,1.264732,1.880398,-0.254039
ES5,1.647918,-0.659135,-0.559496,-0.942793,-0.806008,0.147963,0.280821,0.802122,-0.342000,-0.073761,...,0.963448,1.822567,-1.881274,2.290713,-0.374365,1.465508,0.824499,1.700572,-1.072722,-0.715958
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,-0.196266,-1.162314,1.213693,1.933026,1.533893,-0.535923,0.664679,-1.499521,1.313865,-0.113262,...,-0.930435,-0.121002,0.268203,-0.501986,-0.340202,-0.819995,-0.163018,-0.847269,0.115015,-0.415638
SNU-407,-0.162459,-1.299035,0.705396,0.063787,-0.086131,-0.538136,-0.131771,-1.295700,1.357186,-0.552781,...,-1.048848,-0.974241,-0.565825,-0.880683,-0.611274,-0.141052,-2.517792,-0.596205,0.010876,-0.309140
SNU-61,-0.357438,-0.744172,1.009094,1.253740,-0.705407,-0.518680,0.580381,-1.007094,1.098923,-0.548713,...,-1.225661,-0.756175,0.586308,-0.980272,-1.008351,0.427593,0.079123,-0.653641,1.074412,-0.547522
SNU-81,-0.340462,-0.401068,-0.403490,0.933954,-0.671707,-0.479090,-0.652520,-1.699752,1.787090,-0.452524,...,-1.109103,-0.812905,0.275427,-0.308895,-1.042723,0.030764,-1.430484,-0.815159,0.549280,-0.035066


In [22]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T.values),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,-0.602444,0.848456,0.373377,-0.810112,-0.766023,-0.937399,-0.105971,-0.204095,-0.335437,0.742093,...,-0.881596,-0.868666,1.133114,-0.636321,-0.210260,-0.450032,0.745070,-0.847492,-0.761067,-0.799309
PFSK-1,-0.683017,0.787960,-0.833001,-0.839798,0.576742,0.687061,0.358900,-0.175815,-0.101935,-0.946320,...,0.178773,-0.967915,0.119348,-0.516498,-0.028462,-1.041315,0.903320,-0.546790,-0.521193,-0.153070
A673,-0.793349,-0.196409,-1.041771,-0.883030,-0.563181,-0.865599,0.483990,-0.060390,0.173691,-0.910453,...,0.162193,-0.684318,0.820759,0.839910,-0.767645,0.640328,1.101291,0.304136,0.541522,-0.823786
ES3,-0.692367,0.237510,-0.878138,-0.960912,-0.531118,-0.625829,0.481919,0.216615,-0.125206,-0.457749,...,0.911204,0.625969,1.055906,1.547852,-0.495929,0.906664,1.381267,0.396879,1.304023,-0.450152
ES5,0.745503,0.259720,-1.036144,-0.998352,-0.931046,-0.555352,0.236577,0.365798,0.328085,-0.732924,...,0.559353,0.846234,-0.811524,1.657637,-0.464089,0.745547,1.759834,0.675342,-0.535592,-0.789908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,-0.678954,-0.097484,0.159471,1.068899,0.479660,-1.001469,0.261481,-0.391942,1.387108,-0.731134,...,-0.722054,-0.522070,0.617437,-0.446646,-0.448409,-0.934970,0.777697,-0.954100,0.144107,-0.592982
SNU-407,-0.705185,-0.212359,-0.206619,-0.282112,-0.511951,-1.060483,0.015284,-0.375777,1.412390,-1.035200,...,-0.848263,-1.119829,0.041473,-0.752411,-0.653737,-0.520424,-1.059182,-0.858883,0.051020,-0.581067
SNU-61,-0.888678,0.014622,-0.088214,0.463201,-0.923424,-1.081974,0.117904,-0.358747,1.093407,-1.068131,...,-0.998469,-1.016752,0.689031,-0.865622,-0.929693,-0.202049,0.822299,-0.935080,0.558446,-0.765920
SNU-81,-0.846005,0.277058,-0.947927,0.323953,-0.874351,-1.033171,-0.141670,-0.509761,1.694551,-0.990290,...,-0.899305,-1.028244,0.582069,-0.374057,-0.921134,-0.415951,-0.234861,-1.003813,0.347066,-0.439882


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [23]:
def min_max_scale(data):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = data[data > 0].fillna(0)
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [24]:
A_dc = min_max_scale(drug_response)
A_dc

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.321121,0.000000,0.000000,0.197224,0.000000,0.000000,0.000000,0.000000,0.036509,0.218317,...,0.0,0.0,0.183955,0.186012,0.412511,0.000000,0.000000,0.289624,0.113644,1.000000
Afuresertib,0.000000,0.000000,0.000000,0.343735,0.346504,1.000000,0.693239,0.000000,0.000000,0.000000,...,0.0,0.0,0.703754,0.136263,0.001466,0.000000,0.000000,0.000000,0.000000,0.063752
Alisertib,0.000000,1.000000,0.124087,0.000000,0.000000,0.000000,0.227070,0.000000,0.162812,0.732631,...,1.0,0.0,0.579587,0.000000,0.000000,0.745223,0.703761,0.142483,0.389752,0.306430
Alpelisib,0.000000,0.000000,0.000000,0.000000,0.421564,0.000000,0.310195,1.000000,0.000000,0.155259,...,0.0,0.0,0.005379,0.000000,0.657620,0.000000,0.000000,0.011956,0.228983,0.237618
Avagacestat,0.182712,0.523474,0.311893,0.000000,0.475954,0.128527,0.000000,0.471576,0.456334,0.000000,...,0.0,0.0,0.748281,0.714536,0.000000,0.000000,0.000000,0.000000,0.325694,0.048769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.000000,0.000000,0.288766,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.862610,...,0.0,0.0,0.398332,0.000000,0.200172,0.000000,0.344363,0.197421,0.194050,0.513774
Venetoclax,0.550395,0.037173,0.482813,0.046123,0.298342,0.000000,0.734852,0.040893,0.000000,0.157140,...,0.0,0.0,0.757210,0.222973,0.000000,0.114720,0.575162,0.193739,0.105724,0.400468
Vinorelbine,0.000000,0.000000,0.000000,0.136250,0.000000,0.527306,0.428781,0.000000,0.000000,0.000000,...,0.0,0.0,1.000000,0.000000,0.081301,0.000000,0.787292,0.273375,0.256019,0.289256
Vismodegib,0.290042,0.000000,0.000000,0.000000,0.029871,0.000000,0.000000,0.566324,0.000000,0.000000,...,0.0,0.0,0.218063,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.185896


In [25]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,0.000000,0.300338,0.240500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.521208,...,0.000000,0.000000,0.509096,0.000000,0.000000,0.000000,0.092732,0.000000,0.000000,0.000000
PFSK-1,0.000000,0.286559,0.000000,0.000000,0.462282,0.328708,0.271308,0.000000,0.000000,0.000000,...,0.147889,0.000000,0.000000,0.000000,0.060563,0.000000,0.202464,0.000000,0.000000,0.029999
A673,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.409765,0.000000,0.000000,0.000000,...,0.140246,0.000000,0.340989,0.417286,0.000000,0.470486,0.321060,0.324786,0.224957,0.000000
ES3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.353046,0.104521,0.000000,0.000000,...,0.556944,0.605366,0.459421,0.721729,0.000000,0.591467,0.449726,0.354911,0.560831,0.000000
ES5,0.270615,0.000000,0.000000,0.000000,0.000000,0.000000,0.116100,0.243969,0.000000,0.000000,...,0.351973,0.755015,0.000000,0.758039,0.000000,0.501547,0.643434,0.507482,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,0.000000,0.000000,0.188813,0.748051,0.427158,0.000000,0.207823,0.000000,0.459164,0.000000,...,0.000000,0.000000,0.228213,0.000000,0.000000,0.000000,0.153040,0.000000,0.045636,0.000000
SNU-407,0.000000,0.000000,0.068583,0.000000,0.000000,0.000000,0.000000,0.000000,0.470826,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010901,0.000000
SNU-61,0.000000,0.000000,0.126623,0.427845,0.000000,0.000000,0.156690,0.000000,0.372695,0.000000,...,0.000000,0.000000,0.328631,0.000000,0.000000,0.051162,0.224431,0.000000,0.287574,0.000000
SNU-81,0.000000,0.000000,0.000000,0.313458,0.000000,0.000000,0.000000,0.000000,0.591877,0.000000,...,0.000000,0.000000,0.220961,0.000000,0.000000,0.000000,0.000000,0.000000,0.157862,0.000000


In [26]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[i["Drug Name"], i["Gene"]] = 1
A_dg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
Afatinib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Afuresertib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alisertib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alpelisib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Avagacestat,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Venetoclax,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Vinorelbine,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Vismodegib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [27]:
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.499758,-0.287872,-0.044328,0.445754,-0.018312,-0.056024,-1.007905,-0.133606,0.039340,0.332709,...,-0.630538,-1.313438,0.462079,0.219408,0.967360,-0.212762,-2.278620,0.570567,0.332668,2.724350
Afuresertib,-0.576528,-1.027199,-1.251724,0.776892,0.483202,1.035221,0.702506,-1.467521,-0.995829,0.000000,...,-3.524001,-2.226865,1.767767,0.160727,0.003439,0.000000,0.000000,0.000000,-0.303296,0.173684
Alisertib,0.000000,1.227995,0.106415,0.000000,-0.067135,0.000000,0.230105,-0.070375,0.175436,1.116507,...,0.747585,-1.310767,1.455871,-0.736767,-0.112505,0.413056,2.098072,0.280696,1.140913,0.834823
Alpelisib,0.000000,-1.081335,-0.328515,0.000000,0.587874,0.000000,0.314342,1.214618,-0.389851,0.236610,...,-0.565499,-1.668067,0.013511,-0.751235,1.542154,-0.815384,-0.349031,0.023554,0.670297,0.647354
Avagacestat,0.284354,0.642823,0.267475,-1.214956,0.663721,0.133054,-0.413084,0.572785,0.491717,0.000000,...,-0.729761,-0.043942,1.879616,0.842820,-0.229219,0.000000,0.000000,0.000000,0.953398,0.132863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.000000,-0.024717,0.247641,0.000000,-0.277552,0.000000,-0.148106,-0.020043,-1.006031,1.314591,...,-1.588061,0.000000,1.000575,-1.115229,0.469415,-1.975273,1.026625,0.388924,0.568039,1.399699
Venetoclax,0.856575,0.045648,0.414053,0.104244,0.416040,-0.285095,0.744676,0.049669,-0.024445,0.239476,...,-2.341882,-1.468818,1.902045,0.263005,-0.077745,0.063586,1.714690,0.381670,0.309484,1.091016
Vinorelbine,-0.189093,-1.092180,-0.439257,0.307946,-0.064701,0.545878,0.434513,-0.945450,-0.099919,-0.240916,...,-1.516742,-1.237351,2.511912,-0.572717,0.190656,-0.305630,2.347096,0.538555,0.749441,0.788034
Vismodegib,0.451390,-0.355878,-0.326051,-0.267893,0.041656,-0.965318,-0.362449,0.687867,-0.345633,0.000000,...,-0.567849,0.000000,0.547754,-0.326945,-0.521560,0.000000,0.000000,0.000000,-0.548637,0.506446


In [28]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  40.18 %
Cell Density:  41.620000000000005 %
Gene Density:  100.0 %


# Similarity

In [29]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [30]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../GDSC2_data/cell_sim.csv")
cell_sim

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
22RV1,1.000000,0.558877,0.383537,0.218290,0.476917,0.537868,0.402155,0.575215,0.483448,0.563269,...,0.138528,0.057204,0.212700,0.512884,0.482895,0.458451,0.105355,0.534960,0.483671,0.258678
23132-87,0.558877,1.000000,0.531469,0.156809,0.538271,0.503205,0.420569,0.595689,0.622697,0.433507,...,0.313277,0.138672,0.100352,0.593245,0.402583,0.501064,0.063353,0.324158,0.382355,0.139415
42-MG-BA,0.383537,0.531469,1.000000,0.117032,0.532218,0.471016,0.441408,0.614337,0.621272,0.346796,...,0.279929,0.160680,0.072667,0.550691,0.393238,0.482752,0.050350,0.295562,0.294535,0.095552
451Lu,0.218290,0.156809,0.117032,1.000000,0.206412,0.249985,0.262241,0.168759,0.159821,0.303621,...,0.025678,0.010811,0.106480,0.146366,0.324346,0.232555,0.056800,0.276947,0.281019,0.146201
639-V,0.476917,0.538271,0.532218,0.206412,1.000000,0.764270,0.492873,0.591446,0.645684,0.374049,...,0.132610,0.075954,0.161016,0.558323,0.584483,0.465035,0.075676,0.413857,0.418123,0.185200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.458451,0.501064,0.482752,0.232555,0.465035,0.462913,0.591714,0.506749,0.595923,0.368913,...,0.280851,0.133440,0.077268,0.532083,0.377580,1.000000,0.042499,0.325266,0.345635,0.091165
ZR-75-30,0.105355,0.063353,0.050350,0.056800,0.075676,0.091917,0.094265,0.071433,0.043657,0.139843,...,0.007478,0.002363,0.188555,0.058594,0.131567,0.042499,1.000000,0.203478,0.156310,0.156759
huH-1,0.534960,0.324158,0.295562,0.276947,0.413857,0.491343,0.343718,0.388032,0.331978,0.591108,...,0.064235,0.024543,0.248962,0.356250,0.468280,0.325266,0.203478,1.000000,0.434481,0.282266
no-10,0.483671,0.382355,0.294535,0.281019,0.418123,0.493461,0.398588,0.384339,0.401107,0.390720,...,0.086587,0.032500,0.213788,0.364669,0.584479,0.345635,0.156310,0.434481,1.000000,0.293488


In [31]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.2817405720328896
Median: 0.27583336835226224


In [32]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../GDSC2_data/gene_sim.csv")
gene_sim

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
A2M,1.000000,0.122510,0.123400,0.088330,0.075220,0.118393,0.116858,0.175571,0.084031,0.089788,...,0.125916,0.119805,0.141814,0.092885,0.157358,0.085962,0.099398,0.130532,0.080746,0.089855
AAED1,0.122510,1.000000,0.103315,0.162484,0.065855,0.187494,0.177476,0.226015,0.082277,0.045829,...,0.158549,0.119975,0.210392,0.114317,0.078720,0.059882,0.100786,0.100737,0.169264,0.154126
ABCB1,0.123400,0.103315,1.000000,0.160006,0.100523,0.112892,0.114793,0.104328,0.167157,0.114300,...,0.118313,0.093226,0.116010,0.097047,0.154011,0.131755,0.093676,0.113404,0.101951,0.105067
ABCC3,0.088330,0.162484,0.160006,1.000000,0.136852,0.155008,0.109648,0.112810,0.246798,0.046345,...,0.098182,0.099522,0.151761,0.124760,0.093997,0.046710,0.099932,0.083201,0.193903,0.226953
ABCG1,0.075220,0.065855,0.100523,0.136852,1.000000,0.082214,0.095874,0.056406,0.188355,0.138101,...,0.095143,0.104194,0.080687,0.167873,0.096808,0.163991,0.113069,0.129339,0.122641,0.149008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF711,0.085962,0.059882,0.131755,0.046710,0.163991,0.075437,0.147640,0.070592,0.116553,0.135689,...,0.202422,0.170409,0.133474,0.152286,0.156069,1.000000,0.141010,0.265314,0.079753,0.106820
ZNF83,0.099398,0.100786,0.093676,0.099932,0.113069,0.149690,0.160597,0.125930,0.087022,0.124461,...,0.156094,0.311728,0.120299,0.163820,0.168165,0.141010,1.000000,0.157693,0.097302,0.111617
ZNF853,0.130532,0.100737,0.113404,0.083201,0.129339,0.097803,0.156404,0.115781,0.121869,0.063756,...,0.212188,0.223951,0.194026,0.176827,0.177194,0.265314,0.157693,1.000000,0.083953,0.123620
ZP3,0.080746,0.169264,0.101951,0.193903,0.122641,0.121772,0.102588,0.131102,0.165964,0.072506,...,0.093924,0.099423,0.122103,0.192523,0.074307,0.079753,0.097302,0.083953,1.000000,0.146863


In [33]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 0.9999999999999999
Mean: 0.1343433810949062
Median: 0.11975286042035213


# NSC to SMILES

In [34]:
convert = dict(
    pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
        ["NAME", "SMILES"]
    ].values
)
SMILES = [convert[i] for i in drug_response.index]
SMILES

['CN(C)CC=CC(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C=C3)F)Cl)OC4CCOC4',
 'CN1C(=C(C=N1)Cl)C2=C(SC(=C2)C(=O)NC(CC3=CC(=CC=C3)F)CN)Cl',
 'COC1=C(C(=CC=C1)F)C2=NCC3=CN=C(N=C3C4=C2C=C(C=C4)Cl)NC5=CC(=C(C=C5)C(=O)O)OC',
 'CC1=C(SC(=N1)NC(=O)N2CCCC2C(=O)N)C3=CC(=NC=C3)C(C)(C)C(F)(F)F',
 'C1=CC(=CC=C1S(=O)(=O)N(CC2=C(C=C(C=C2)C3=NOC=N3)F)C(CCC(F)(F)F)C(=O)N)Cl',
 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)C=CC4=CC=CC=N4',
 'B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN=C2)(O)O',
 'CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC',
 'CCOC(=O)NC1=CC(=NN2C1=NN=C2C)C3=CC(=C(C=C3)C)NS(=O)(=O)C',
 'C1COCCN1C2=NC(=NC(=C2)C3=CN=C(C=C3C(F)(F)F)N)N4CCOCC4',
 'CCC1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=C4C3=C2)O',
 'CC1=CC2=C(N1)C=CC(=C2F)OC3=NC=NC4=CC(=C(C=C43)OC)OCCCN5CCCC5',
 'CC(C1=C(C=CC(=C1Cl)F)Cl)OC2=C(N=CC(=C2)C3=CN(N=C3)C4CCNCC4)N',
 'C1CNP(=O)(OC1)N(CCCl)CCCl',
 'CC(C)(C)C1=NC(=C(S1)C2=NC(=NC=C2)N)C3=C(C(=CC=C3)NS(=O)(=O)C4=C(C=CC=C4F)F)F',
 'CC(C)(C#N)C1=CC=C(C=C1)N2C3=C4C=C(

In [35]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [36]:
mfp = get_morgan_fingerprint(SMILES)
mfp = pd.DataFrame(mfp, index=drug_response.index)

In [37]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim.to_csv("../GDSC2_data/drug_sim.csv")
drug_sim

,Afatinib,Afuresertib,Alisertib,Alpelisib,Avagacestat,Axitinib,Bortezomib,Bosutinib,Bromosporine,Buparlisib,...,Tozasertib,Trametinib,Tretinoin,Ulixertinib,Uprosertib,Veliparib,Venetoclax,Vinorelbine,Vismodegib,Vorinostat
Afatinib,1.000000,0.362041,0.418190,0.343380,0.362041,0.374497,0.349597,0.411939,0.368268,0.374497,...,0.362041,0.349597,0.380730,0.380730,0.418190,0.399447,0.201212,0.121559,0.449491,0.480868
Afuresertib,0.362041,1.000000,0.430702,0.443225,0.449491,0.449491,0.537541,0.411939,0.455761,0.487153,...,0.411939,0.399447,0.443225,0.543853,0.798891,0.499732,0.176649,0.195066,0.524926,0.569134
Alisertib,0.418190,0.430702,1.000000,0.362041,0.405691,0.405691,0.455761,0.418190,0.374497,0.418190,...,0.393205,0.380730,0.411939,0.449491,0.436962,0.468308,0.195066,0.176649,0.506026,0.524926
Alpelisib,0.343380,0.443225,0.362041,1.000000,0.430702,0.455761,0.468308,0.355818,0.462033,0.493441,...,0.480868,0.393205,0.462033,0.411939,0.436962,0.506026,0.231982,0.188924,0.480868,0.537541
Avagacestat,0.362041,0.449491,0.405691,0.430702,1.000000,0.399447,0.424444,0.337165,0.443225,0.449491,...,0.349597,0.312337,0.418190,0.430702,0.493441,0.449491,0.238145,0.133780,0.487153,0.518623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.399447,0.499732,0.468308,0.506026,0.449491,0.562809,0.524926,0.436962,0.493441,0.524926,...,0.499732,0.436962,0.607147,0.531232,0.480868,1.000000,0.250480,0.343380,0.550169,0.645271
Venetoclax,0.201212,0.176649,0.195066,0.231982,0.238145,0.238145,0.201212,0.225822,0.231982,0.238145,...,0.225822,0.139896,0.219665,0.231982,0.207360,0.250480,1.000000,0.000000,0.287558,0.256652
Vinorelbine,0.121559,0.195066,0.176649,0.188924,0.133780,0.244311,0.219665,0.219665,0.225822,0.195066,...,0.207360,0.170516,0.262828,0.213511,0.188924,0.343380,0.000000,1.000000,0.231982,0.349597
Vismodegib,0.449491,0.524926,0.506026,0.480868,0.487153,0.588126,0.550169,0.436962,0.543853,0.499732,...,0.487153,0.436962,0.506026,0.543853,0.518623,0.550169,0.287558,0.231982,1.000000,0.658004


In [38]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.41791642392394457
Median: 0.4244444296422518


# Unified Graph

In [39]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,Afatinib,Afuresertib,Alisertib,Alpelisib,Avagacestat,Axitinib,Bortezomib,Bosutinib,Bromosporine,Buparlisib,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
Afatinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Afuresertib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Alisertib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Alpelisib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Avagacestat,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF711,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF83,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
A_dc

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.321121,0.000000,0.000000,0.197224,0.000000,0.000000,0.000000,0.000000,0.036509,0.218317,...,0.0,0.0,0.183955,0.186012,0.412511,0.000000,0.000000,0.289624,0.113644,1.000000
Afuresertib,0.000000,0.000000,0.000000,0.343735,0.346504,1.000000,0.693239,0.000000,0.000000,0.000000,...,0.0,0.0,0.703754,0.136263,0.001466,0.000000,0.000000,0.000000,0.000000,0.063752
Alisertib,0.000000,1.000000,0.124087,0.000000,0.000000,0.000000,0.227070,0.000000,0.162812,0.732631,...,1.0,0.0,0.579587,0.000000,0.000000,0.745223,0.703761,0.142483,0.389752,0.306430
Alpelisib,0.000000,0.000000,0.000000,0.000000,0.421564,0.000000,0.310195,1.000000,0.000000,0.155259,...,0.0,0.0,0.005379,0.000000,0.657620,0.000000,0.000000,0.011956,0.228983,0.237618
Avagacestat,0.182712,0.523474,0.311893,0.000000,0.475954,0.128527,0.000000,0.471576,0.456334,0.000000,...,0.0,0.0,0.748281,0.714536,0.000000,0.000000,0.000000,0.000000,0.325694,0.048769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.000000,0.000000,0.288766,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.862610,...,0.0,0.0,0.398332,0.000000,0.200172,0.000000,0.344363,0.197421,0.194050,0.513774
Venetoclax,0.550395,0.037173,0.482813,0.046123,0.298342,0.000000,0.734852,0.040893,0.000000,0.157140,...,0.0,0.0,0.757210,0.222973,0.000000,0.114720,0.575162,0.193739,0.105724,0.400468
Vinorelbine,0.000000,0.000000,0.000000,0.136250,0.000000,0.527306,0.428781,0.000000,0.000000,0.000000,...,0.0,0.0,1.000000,0.000000,0.081301,0.000000,0.787292,0.273375,0.256019,0.289256
Vismodegib,0.290042,0.000000,0.000000,0.000000,0.029871,0.000000,0.000000,0.566324,0.000000,0.000000,...,0.0,0.0,0.218063,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.185896


In [41]:
A_cg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,0.000000,0.300338,0.240500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.521208,...,0.000000,0.000000,0.509096,0.000000,0.000000,0.000000,0.092732,0.000000,0.000000,0.000000
PFSK-1,0.000000,0.286559,0.000000,0.000000,0.462282,0.328708,0.271308,0.000000,0.000000,0.000000,...,0.147889,0.000000,0.000000,0.000000,0.060563,0.000000,0.202464,0.000000,0.000000,0.029999
A673,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.409765,0.000000,0.000000,0.000000,...,0.140246,0.000000,0.340989,0.417286,0.000000,0.470486,0.321060,0.324786,0.224957,0.000000
ES3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.353046,0.104521,0.000000,0.000000,...,0.556944,0.605366,0.459421,0.721729,0.000000,0.591467,0.449726,0.354911,0.560831,0.000000
ES5,0.270615,0.000000,0.000000,0.000000,0.000000,0.000000,0.116100,0.243969,0.000000,0.000000,...,0.351973,0.755015,0.000000,0.758039,0.000000,0.501547,0.643434,0.507482,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,0.000000,0.000000,0.188813,0.748051,0.427158,0.000000,0.207823,0.000000,0.459164,0.000000,...,0.000000,0.000000,0.228213,0.000000,0.000000,0.000000,0.153040,0.000000,0.045636,0.000000
SNU-407,0.000000,0.000000,0.068583,0.000000,0.000000,0.000000,0.000000,0.000000,0.470826,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010901,0.000000
SNU-61,0.000000,0.000000,0.126623,0.427845,0.000000,0.000000,0.156690,0.000000,0.372695,0.000000,...,0.000000,0.000000,0.328631,0.000000,0.000000,0.051162,0.224431,0.000000,0.287574,0.000000
SNU-81,0.000000,0.000000,0.000000,0.313458,0.000000,0.000000,0.000000,0.000000,0.591877,0.000000,...,0.000000,0.000000,0.220961,0.000000,0.000000,0.000000,0.000000,0.000000,0.157862,0.000000


In [42]:
A_dg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
Afatinib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Afuresertib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alisertib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alpelisib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Avagacestat,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Venetoclax,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Vinorelbine,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Vismodegib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [43]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.321121,0.000000,0.000000,0.197224,0.000000,0.000000,0.000000,0.000000,0.036509,0.218317,...,0.0,0.0,0.183955,0.186012,0.412511,0.000000,0.000000,0.289624,0.113644,1.000000
Afuresertib,0.000000,0.000000,0.000000,0.343735,0.346504,1.000000,0.693239,0.000000,0.000000,0.000000,...,0.0,0.0,0.703754,0.136263,0.001466,0.000000,0.000000,0.000000,0.000000,0.063752
Alisertib,0.000000,1.000000,0.124087,0.000000,0.000000,0.000000,0.227070,0.000000,0.162812,0.732631,...,1.0,0.0,0.579587,0.000000,0.000000,0.745223,0.703761,0.142483,0.389752,0.306430
Alpelisib,0.000000,0.000000,0.000000,0.000000,0.421564,0.000000,0.310195,1.000000,0.000000,0.155259,...,0.0,0.0,0.005379,0.000000,0.657620,0.000000,0.000000,0.011956,0.228983,0.237618
Avagacestat,0.182712,0.523474,0.311893,0.000000,0.475954,0.128527,0.000000,0.471576,0.456334,0.000000,...,0.0,0.0,0.748281,0.714536,0.000000,0.000000,0.000000,0.000000,0.325694,0.048769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Veliparib,0.000000,0.000000,0.288766,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.862610,...,0.0,0.0,0.398332,0.000000,0.200172,0.000000,0.344363,0.197421,0.194050,0.513774
Venetoclax,0.550395,0.037173,0.482813,0.046123,0.298342,0.000000,0.734852,0.040893,0.000000,0.157140,...,0.0,0.0,0.757210,0.222973,0.000000,0.114720,0.575162,0.193739,0.105724,0.400468
Vinorelbine,0.000000,0.000000,0.000000,0.136250,0.000000,0.527306,0.428781,0.000000,0.000000,0.000000,...,0.0,0.0,1.000000,0.000000,0.081301,0.000000,0.787292,0.273375,0.256019,0.289256
Vismodegib,0.290042,0.000000,0.000000,0.000000,0.029871,0.000000,0.000000,0.566324,0.000000,0.000000,...,0.0,0.0,0.218063,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.185896


In [45]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

np.save(
    "../GDSC2_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

np.save(
    "../GDSC2_data/edge_idxs.npy",
    edge_index,
)

np.save(
    "../GDSC2_data/edge_attr.npy",
    edge_attr,
)